# Practical 1 — Text Preprocessing & Tokenization

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To perform text preprocessing (case normalization, punctuation/number/whitespace removal) and tokenization (sentence-level and word-level, comparing regex, NLTK, and spaCy) on a small sample review corpus.

## Theory

**Text preprocessing** is the set of steps used to turn raw, noisy text into a clean, consistent form before any downstream NLP task (classification, vectorization, etc.). Raw text usually contains inconsistencies that don't carry useful signal — mixed casing, punctuation, stray digits, extra whitespace — which can otherwise cause the same word to be treated as different tokens (e.g. `"Great!!"` vs `"great"`).

Common preprocessing steps:
- **Lowercasing** — normalizes case so `"Great"` and `"great"` are treated identically.
- **Punctuation/special character removal** — strips symbols that usually don't carry semantic meaning on their own.
- **Number removal** — optional; useful when digits are noise (e.g. ratings like `7/10`) but harmful when numbers matter (e.g. dates, quantities).
- **Whitespace normalization** — collapses multiple spaces/newlines into one.

**Tokenization** is the process of splitting text into smaller units (tokens) — typically sentences or words — that downstream algorithms operate on.
- **Sentence tokenization** splits a document into sentences (not just on `.` — has to handle abbreviations, decimals, etc.).
- **Word tokenization** splits a sentence into words/punctuation tokens. Different tokenizers disagree on edge cases:
  - A **regex tokenizer** (`\w+`) is fast but naive — it can't tell `"don't"` should stay together or split into `do` + `n't`.
  - **NLTK's** word tokenizer follows the Penn Treebank conventions (splits contractions, separates punctuation).
  - **spaCy's** tokenizer is rule-based *and* model-informed, and tends to handle contractions and edge cases more consistently.

Comparing all three on the same input makes these differences concrete rather than theoretical.

## Algorithm

1. Load the sample review dataset (`datasets/sample_reviews.csv`).
2. For each review, apply the cleaning pipeline: lowercase → remove special characters → remove numbers → collapse whitespace.
3. Print a before/after comparison for a few reviews to inspect what changed.
4. Apply sentence tokenization (NLTK) to a multi-sentence example.
5. Apply word tokenization three ways (regex, NLTK, spaCy) to the same sentence and compare outputs.
6. Run the full pipeline (clean → tokenize) across the whole dataset and compute basic summary stats (token counts, vocabulary size).
7. Record actual output and write observations/conclusion based on what you see.

In [1]:
import sys, os
sys.path.append(os.path.abspath("../python"))

import pandas as pd
import nltk

# One-time downloads — safe to re-run, NLTK skips if already present
nltk.download("punkt")
nltk.download("punkt_tab")

import preprocessing
import tokenizer

pd.set_option("display.max_colwidth", None)


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Step 1 — Load the dataset

In [2]:
df = pd.read_csv("../datasets/sample_reviews.csv")
print(f"Loaded {len(df)} reviews")
df.head()


Loaded 15 reviews


,id,review
0,1,This movie was ABSOLUTELY fantastic!!! I've never seen anything like it before.
1,2,"Worst film of 2026. Don't waste your $12 on a ticket, I promise you won't like it."
2,3,"A solid 7/10 - great visuals, but the plot dragged on for way too long..."
3,4,I can't believe how good the acting was!! Robert De Niro really outdid himself this time.
4,5,"meh. it was fine i guess?? nothing special, wouldn't watch again tbh"


### Step 2 — Preprocessing: before vs after

In [3]:
for i in range(3):
    raw = df.loc[i, "review"]
    cleaned = preprocessing.clean_text(raw)
    print(f"RAW:     {raw}")
    print(f"CLEANED: {cleaned}")
    print("-" * 60)


RAW:     This movie was ABSOLUTELY fantastic!!! I've never seen anything like it   before.
CLEANED: this movie was absolutely fantastic ive never seen anything like it before
------------------------------------------------------------
RAW:     Worst   film of 2026. Don't waste your $12 on a ticket, I promise you won't like it.
CLEANED: worst film of dont waste your on a ticket i promise you wont like it
------------------------------------------------------------
RAW:     A solid 7/10 - great visuals, but the plot dragged on for way too long...
CLEANED: a solid great visuals but the plot dragged on for way too long
------------------------------------------------------------


### Step 3 — Sentence tokenization (NLTK)

In [4]:
sample = "This movie was great. I especially loved the soundtrack! Would I watch it again? Probably not."
sentences = tokenizer.nltk_sentence_tokenize(sample)
for s in sentences:
    print(s)


This movie was great.
I especially loved the soundtrack!
Would I watch it again?
Probably not.


### Step 4 — Word tokenization: regex vs NLTK vs spaCy

In [5]:
test_sentence = "I can't believe how good the acting was!! Robert De Niro really outdid himself."
results = tokenizer.compare_tokenizers(test_sentence)

for name, tokens in results.items():
    print(f"{name.upper():>6}: {tokens}")


 REGEX: ['I', 'can', 't', 'believe', 'how', 'good', 'the', 'acting', 'was', 'Robert', 'De', 'Niro', 'really', 'outdid', 'himself']
  NLTK: ['I', 'ca', "n't", 'believe', 'how', 'good', 'the', 'acting', 'was', '!', '!', 'Robert', 'De', 'Niro', 'really', 'outdid', 'himself', '.']
 SPACY: ['I', 'ca', "n't", 'believe', 'how', 'good', 'the', 'acting', 'was', '!', '!', 'Robert', 'De', 'Niro', 'really', 'outdid', 'himself', '.']


**Look closely at how each tokenizer handles `"can't"` and the punctuation (`!!`) — that's the comparison this step is meant to surface. Note what you actually observe in the Output section below.**

### Step 5 — Full pipeline across the dataset

In [6]:
all_tokens = []

for review in df["review"]:
    cleaned = preprocessing.clean_text(review)
    tokens = tokenizer.regex_word_tokenize(cleaned)
    all_tokens.append(tokens)

df["tokens"] = all_tokens
df["token_count"] = df["tokens"].apply(len)

vocab = set(t for tokens in all_tokens for t in tokens)

print(f"Average tokens per review: {df['token_count'].mean():.2f}")
print(f"Vocabulary size (unique tokens): {len(vocab)}")
df[["review", "token_count"]]


Average tokens per review: 12.93
Vocabulary size (unique tokens): 136


,review,token_count
0,This movie was ABSOLUTELY fantastic!!! I've never seen anything like it before.,12
1,"Worst film of 2026. Don't waste your $12 on a ticket, I promise you won't like it.",15
2,"A solid 7/10 - great visuals, but the plot dragged on for way too long...",13
3,I can't believe how good the acting was!! Robert De Niro really outdid himself this time.,16
4,"meh. it was fine i guess?? nothing special, wouldn't watch again tbh",12
5,The director's 3rd project is by FAR his best work -- a must watch in theaters.,15
6,Two hours and 15 mins of pure boredom. 2/10 would not recommend to anyone.,12
7,"WOW!!! Best film I've seen in years, hands down. 10/10 no notes.",11
8,"It's okay... not great, not terrible. Somewhere in the middle I'd say.",12
9,Ticket prices are insane these days ($18.50!) but this one was worth every penny.,13


---
## Output

*Run every cell above top to bottom, then paste or describe your actual output here — the printed before/after text, the tokenizer comparison, the average token count and vocabulary size. Don't fill this in until you've actually run it.*


## Observations & Conclusion

Answer these based on what you actually saw when you ran the notebook:

- Which preprocessing step changed the text the most for this dataset — and did anything get removed that you didn't expect (e.g. useful info lost when numbers were stripped)?
- Where did the regex, NLTK, and spaCy tokenizers disagree? Was it only on contractions, or did punctuation/numbers cause differences too?
- What was the average token count and vocabulary size across the 15 reviews — does that number make sense given the dataset size?
- If you were preparing this text for a downstream task (e.g. sentiment classification), would you keep numbers? Why or why not?

*(Write 4-6 sentences here in your own words once you've run the notebook.)*


---
## Viva Prep — Practice Questions

Study aid for the oral exam — work through these yourself rather than memorizing the answers verbatim, since you'll need to explain the reasoning, not recite it.

1. **Why is lowercasing usually done before other preprocessing steps?**
   So that case differences don't cause the same word to be treated as two different tokens later (e.g. "Movie" and "movie" should count as one type when building a vocabulary).

2. **Give an example where removing numbers would lose important information.**
   E.g. a review mentioning "2 stars" vs "10 stars" — stripping digits would make both reviews look identical on that dimension, which matters if you were trying to correlate text with a numeric rating.

3. **What's the difference between sentence tokenization and word tokenization?**
   Sentence tokenization splits a document into sentences (has to reason about periods that aren't sentence boundaries, like abbreviations); word tokenization splits a sentence (or document) into individual word/punctuation tokens.

4. **Why might NLTK and spaCy tokenize a contraction like "don't" differently from a plain regex tokenizer?**
   A regex tokenizer built on `\w+` just matches alphanumeric runs, so it has no linguistic rule for contractions and may output "dont" as one token. NLTK and spaCy apply tokenization rules (and in spaCy's case, a trained model) that recognize "don't" as two meaningful units.

5. **Why do we compute vocabulary size, and what does a small vs large vocabulary tell you about a dataset?**
   Vocabulary size (count of unique tokens) is a basic measure of lexical diversity. A small vocabulary relative to the number of documents suggests repetitive/similar language; a large one suggests more varied vocabulary — this affects decisions later, e.g. how much dimensionality a Bag-of-Words/TF-IDF representation will have.
